# Day 7: Mini Project — Multi-Algorithm Comparison

## Overview

Apply all 5 algorithms (Decision Trees, Random Forests, XGBoost, SVM, KNN) to the same classification problem. Compare accuracy, training time, prediction time, and interpretability. Build your model selection intuition.

## Project Goals

- Train and tune 5 algorithms on a real dataset
- Compare on multiple criteria (speed, accuracy, interpretability)
- Understand trade-offs between algorithms
- Make informed decisions about algorithm selection

## Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
import xgboost as xgb

np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load and Prepare Data

In [ ]:
# Load breast cancer dataset
data = load_breast_cancer()
X = data.data
y = data.target

print(f"Dataset shape: {X.shape}")
print(f"Target distribution: {np.unique(y, return_counts=True)}")
print(f"Features: {data.feature_names}")

# Split into train/validation/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)

# Scale for distance-based models
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"\nTrain: {X_train.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test: {X_test.shape}")

## 2. Algorithm 1: Decision Tree

In [ ]:
# Grid search for Decision Tree
dt_params = {'max_depth': [3, 5, 7, 10], 'min_samples_leaf': [1, 5, 10]}
dt_grid = GridSearchCV(DecisionTreeClassifier(random_state=42), dt_params, cv=5, n_jobs=-1)

start = time.time()
dt_grid.fit(X_train, y_train)
dt_time = time.time() - start

dt_pred_start = time.time()
dt_pred = dt_grid.predict(X_test)
dt_pred_time = time.time() - dt_pred_start

dt_results = {
    'Algorithm': 'Decision Tree',
    'Best Params': str(dt_grid.best_params_),
    'Train Time': dt_time,
    'Pred Time': dt_pred_time,
    'Train Acc': dt_grid.best_score_,
    'Test Acc': accuracy_score(y_test, dt_pred),
    'F1': f1_score(y_test, dt_pred),
    'AUC': roc_auc_score(y_test, dt_grid.decision_function(X_test)) if hasattr(dt_grid, 'decision_function') else roc_auc_score(y_test, dt_grid.predict_proba(X_test)[:, 1])
}

print(f"Decision Tree Results:")
print(f"Best Params: {dt_results['Best Params']}")
print(f"Train Accuracy (CV): {dt_results['Train Acc']:.4f}")
print(f"Test Accuracy: {dt_results['Test Acc']:.4f}")
print(f"F1 Score: {dt_results['F1']:.4f}")
print(f"Train Time: {dt_results['Train Time']:.4f}s | Pred Time: {dt_results['Pred Time']:.4f}s")

## 3. Algorithm 2: Random Forest

In [ ]:
# Grid search for Random Forest
rf_params = {'n_estimators': [50, 100, 200], 'max_depth': [5, 10, 15]}
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1), rf_params, cv=5, n_jobs=-1)

start = time.time()
rf_grid.fit(X_train, y_train)
rf_time = time.time() - start

rf_pred_start = time.time()
rf_pred = rf_grid.predict(X_test)
rf_pred_time = time.time() - rf_pred_start

rf_results = {
    'Algorithm': 'Random Forest',
    'Best Params': str(rf_grid.best_params_),
    'Train Time': rf_time,
    'Pred Time': rf_pred_time,
    'Train Acc': rf_grid.best_score_,
    'Test Acc': accuracy_score(y_test, rf_pred),
    'F1': f1_score(y_test, rf_pred),
    'AUC': roc_auc_score(y_test, rf_grid.predict_proba(X_test)[:, 1])
}

print(f"Random Forest Results:")
print(f"Best Params: {rf_results['Best Params']}")
print(f"Train Accuracy (CV): {rf_results['Train Acc']:.4f}")
print(f"Test Accuracy: {rf_results['Test Acc']:.4f}")
print(f"F1 Score: {rf_results['F1']:.4f}")
print(f"Train Time: {rf_results['Train Time']:.4f}s | Pred Time: {rf_results['Pred Time']:.4f}s")

## 4. Algorithm 3: XGBoost

In [ ]:
# Grid search for XGBoost
xgb_params = {'learning_rate': [0.05, 0.1], 'max_depth': [3, 5, 7], 'n_estimators': [100, 200]}
xgb_grid = GridSearchCV(xgb.XGBClassifier(random_state=42, verbosity=0), xgb_params, cv=5, n_jobs=-1)

start = time.time()
xgb_grid.fit(X_train, y_train)
xgb_time = time.time() - start

xgb_pred_start = time.time()
xgb_pred = xgb_grid.predict(X_test)
xgb_pred_time = time.time() - xgb_pred_start

xgb_results = {
    'Algorithm': 'XGBoost',
    'Best Params': str({k: v for k, v in xgb_grid.best_params_.items()}),
    'Train Time': xgb_time,
    'Pred Time': xgb_pred_time,
    'Train Acc': xgb_grid.best_score_,
    'Test Acc': accuracy_score(y_test, xgb_pred),
    'F1': f1_score(y_test, xgb_pred),
    'AUC': roc_auc_score(y_test, xgb_grid.predict_proba(X_test)[:, 1])
}

print(f"XGBoost Results:")
print(f"Best Params: {xgb_results['Best Params']}")
print(f"Train Accuracy (CV): {xgb_results['Train Acc']:.4f}")
print(f"Test Accuracy: {xgb_results['Test Acc']:.4f}")
print(f"F1 Score: {xgb_results['F1']:.4f}")
print(f"Train Time: {xgb_results['Train Time']:.4f}s | Pred Time: {xgb_results['Pred Time']:.4f}s")

## 5. Algorithm 4: SVM

In [ ]:
# Grid search for SVM
svm_params = {'C': [0.1, 1, 10], 'kernel': ['rbf'], 'gamma': ['scale', 'auto']}
svm_grid = GridSearchCV(SVC(probability=True, random_state=42), svm_params, cv=5, n_jobs=-1)

start = time.time()
svm_grid.fit(X_train_scaled, y_train)
svm_time = time.time() - start

svm_pred_start = time.time()
svm_pred = svm_grid.predict(X_test_scaled)
svm_pred_time = time.time() - svm_pred_start

svm_results = {
    'Algorithm': 'SVM (RBF)',
    'Best Params': str(svm_grid.best_params_),
    'Train Time': svm_time,
    'Pred Time': svm_pred_time,
    'Train Acc': svm_grid.best_score_,
    'Test Acc': accuracy_score(y_test, svm_pred),
    'F1': f1_score(y_test, svm_pred),
    'AUC': roc_auc_score(y_test, svm_grid.predict_proba(X_test_scaled)[:, 1])
}

print(f"SVM Results:")
print(f"Best Params: {svm_results['Best Params']}")
print(f"Train Accuracy (CV): {svm_results['Train Acc']:.4f}")
print(f"Test Accuracy: {svm_results['Test Acc']:.4f}")
print(f"F1 Score: {svm_results['F1']:.4f}")
print(f"Train Time: {svm_results['Train Time']:.4f}s | Pred Time: {svm_results['Pred Time']:.4f}s")

## 6. Algorithm 5: KNN

In [ ]:
# Grid search for KNN
knn_params = {'n_neighbors': [3, 5, 7, 10]}
knn_grid = GridSearchCV(KNeighborsClassifier(), knn_params, cv=5, n_jobs=-1)

start = time.time()
knn_grid.fit(X_train_scaled, y_train)
knn_time = time.time() - start

knn_pred_start = time.time()
knn_pred = knn_grid.predict(X_test_scaled)
knn_pred_time = time.time() - knn_pred_start

knn_results = {
    'Algorithm': 'KNN',
    'Best Params': str(knn_grid.best_params_),
    'Train Time': knn_time,
    'Pred Time': knn_pred_time,
    'Train Acc': knn_grid.best_score_,
    'Test Acc': accuracy_score(y_test, knn_pred),
    'F1': f1_score(y_test, knn_pred),
    'AUC': roc_auc_score(y_test, knn_grid.predict_proba(X_test_scaled)[:, 1])
}

print(f"KNN Results:")
print(f"Best Params: {knn_results['Best Params']}")
print(f"Train Accuracy (CV): {knn_results['Train Acc']:.4f}")
print(f"Test Accuracy: {knn_results['Test Acc']:.4f}")
print(f"F1 Score: {knn_results['F1']:.4f}")
print(f"Train Time: {knn_results['Train Time']:.4f}s | Pred Time: {knn_results['Pred Time']:.4f}s")

## 7. Comparison & Analysis

In [ ]:
# Compile results
results_df = pd.DataFrame([
    {**dt_results, 'Train Time': f"{dt_results['Train Time']:.4f}", 'Pred Time': f"{dt_results['Pred Time']:.6f}"},
    {**rf_results, 'Train Time': f"{rf_results['Train Time']:.4f}", 'Pred Time': f"{rf_results['Pred Time']:.6f}"},
    {**xgb_results, 'Train Time': f"{xgb_results['Train Time']:.4f}", 'Pred Time': f"{xgb_results['Pred Time']:.6f}"},
    {**svm_results, 'Train Time': f"{svm_results['Train Time']:.4f}", 'Pred Time': f"{svm_results['Pred Time']:.6f}"},
    {**knn_results, 'Train Time': f"{knn_results['Train Time']:.4f}", 'Pred Time': f"{knn_results['Pred Time']:.6f}"}
])

print("\n" + "="*100)
print("ALGORITHM COMPARISON SUMMARY")
print("="*100)
print(results_df.to_string(index=False))
print("="*100)

# Visualize accuracy
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Accuracy comparison
ax = axes[0, 0]
algs = results_df['Algorithm'].values
test_accs = [float(dt_results['Test Acc']), float(rf_results['Test Acc']), 
              float(xgb_results['Test Acc']), float(svm_results['Test Acc']), 
              float(knn_results['Test Acc'])]
ax.bar(algs, test_accs)
ax.set_ylabel('Test Accuracy')
ax.set_title('Algorithm Performance: Test Accuracy')
ax.set_ylim([0.9, 1.0])
for i, v in enumerate(test_accs):
    ax.text(i, v + 0.002, f'{v:.4f}', ha='center')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# F1 scores
ax = axes[0, 1]
f1_scores = [float(dt_results['F1']), float(rf_results['F1']), 
              float(xgb_results['F1']), float(svm_results['F1']), 
              float(knn_results['F1'])]
ax.bar(algs, f1_scores, color='orange')
ax.set_ylabel('F1 Score')
ax.set_title('Algorithm Performance: F1 Score')
ax.set_ylim([0.9, 1.0])
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Training time
ax = axes[1, 0]
train_times = [dt_results['Train Time'], rf_results['Train Time'], 
                xgb_results['Train Time'], svm_results['Train Time'], 
                knn_results['Train Time']]
ax.bar(algs, train_times, color='green')
ax.set_ylabel('Time (seconds)')
ax.set_title('Training Time')
ax.set_yscale('log')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Prediction time
ax = axes[1, 1]
pred_times = [dt_results['Pred Time'], rf_results['Pred Time'], 
               xgb_results['Pred Time'], svm_results['Pred Time'], 
               knn_results['Pred Time']]
ax.bar(algs, pred_times, color='red')
ax.set_ylabel('Time (seconds)')
ax.set_title('Prediction Time')
ax.set_yscale('log')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

## Analysis & Recommendations

In [ ]:
print("\n" + "="*100)
print("ANALYSIS & RECOMMENDATIONS")
print("="*100)

print(f"\n1. ACCURACY WINNER: {max(results_df['Algorithm'].values, key=lambda x: [dt_results, rf_results, xgb_results, svm_results, knn_results][[dt_results, rf_results, xgb_results, svm_results, knn_results][results_df['Algorithm'].values.tolist().index(x)]]['Test Acc']) if results_df['Algorithm'].values.tolist() else 'N/A'}")
print(f"   → XGBoost and Random Forest typically perform best on tabular data")

print(f"\n2. SPEED (Training):")
fast_train_idx = np.argmin(train_times)
print(f"   → {algs[fast_train_idx]} is fastest ({train_times[fast_train_idx]:.4f}s)")

print(f"\n3. SPEED (Prediction):")
fast_pred_idx = np.argmin(pred_times)
print(f"   → {algs[fast_pred_idx]} is fastest ({pred_times[fast_pred_idx]:.6f}s)")

print(f"\n4. TRADE-OFF ANALYSIS:")
print(f"   → Trees (DT, RF): Fast, good accuracy, interpretable, no scaling needed")
print(f"   → XGBoost: Excellent accuracy, moderate speed, black box")
print(f"   → SVM: Moderate speed (scaling required), good for small-medium data")
print(f"   → KNN: Instant training, slow prediction, sensitive to scaling")

print(f"\n5. RECOMMENDATION FOR PRODUCTION:")
print(f"   → Use XGBoost or Random Forest for best accuracy")
print(f"   → Use Decision Tree if interpretability is critical")
print(f"   → Use KNN only for very small datasets")
print(f"   → Always use validation set to monitor for overfitting")

print("\n" + "="*100)